# mPES — Colab Pro+ OOD Benchmark Sweep

Runs the full **7 models × 22 scenarios = 154 cells** OOD benchmark on Colab Pro+ background execution.

Workflow:
1. Edit `PKG_FILTER`, `GIT_BRANCH` in cell 1 (defaults are usually fine).
2. Run all cells (Runtime → Run all).
3. Click **Runtime → Manage sessions → Background execution** (Pro+ only).
4. Close the browser. Re-open the notebook later to inspect `general/results/`.

Drive layout:
```
/content/drive/MyDrive/mPES/_benchmark/
    inputs/<pkg>/             # optional: pre-staged trained artefacts
    results/                  # mirrored from general/results/
    orchestrate.log
    aggregate.log
    plot.log
    report.log
```

If `_benchmark/inputs/<pkg>/` is empty for a package, the launcher
falls back to the most recent `<DATE>_BAYESIAN_OPT/inputs/` folder
under that package's Drive directory.

In [ ]:
"""mPES Colab benchmark launcher (cell 1: parameters and environment)."""
# pylint: disable=missing-module-docstring,reimported,ungrouped-imports,wrong-import-position
# ============================================================================
# Cell 1 — PARAMETERS (the only cell you normally edit)
# ============================================================================
PKG_FILTER  = ''                             # '' = all 7 pkgs; or e.g. 'pes_dqn'
GIT_REPO    = 'https://github.com/Maximiliano0/mPES_2026.git'
GIT_BRANCH  = 'models_improvement'
USE_GPU     = 0                              # 0 = CPU (recommended for inference)

import os
os.environ['PKG_FILTER'] = PKG_FILTER
os.environ['GIT_REPO']   = GIT_REPO
os.environ['GIT_BRANCH'] = GIT_BRANCH
os.environ['MPES_USE_GPU'] = str(USE_GPU)
print(f'[CELL 1] PKG_FILTER={PKG_FILTER!r} GIT_BRANCH={GIT_BRANCH!r} USE_GPU={USE_GPU}')

In [ ]:
# ============================================================================
# Cell 2 — MOUNT GOOGLE DRIVE
# ============================================================================
# pyright: reportMissingImports=false
# pylint: disable=import-error,no-name-in-module
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive', force_remount=False)

In [ ]:
# pylint: disable=missing-module-docstring,reimported,ungrouped-imports,wrong-import-position
import subprocess  # noqa: E402
import os          # noqa: E402

# ============================================================================
# Cell 3 — CLONE/UPDATE REPO + INSTALL DEPS
# ============================================================================
_script = '''
set -euo pipefail
REPO_DIR=/content/Win_mPES
if [[ ! -d "$REPO_DIR/.git" ]]; then
    git clone --branch "$GIT_BRANCH" "$GIT_REPO" "$REPO_DIR"
else
    cd "$REPO_DIR" && git fetch --all && git checkout "$GIT_BRANCH" && git pull
fi
cd "$REPO_DIR"
bash utils/colab/setup_colab.sh
'''
subprocess.run(_script, shell=True, executable='/bin/bash', check=True, env=os.environ.copy())

In [ ]:
# pylint: disable=missing-module-docstring,reimported,ungrouped-imports,wrong-import-position
import subprocess  # noqa: E402
import os          # noqa: E402

# ============================================================================
# Cell 4 — LAUNCH the benchmark sweep (foreground, resumable)
# ============================================================================
_script = '''
set -uo pipefail
cd /content/Win_mPES
bash general/run_colab_bench.sh "$PKG_FILTER"
'''
_proc = subprocess.run(
    _script, shell=True, executable='/bin/bash',
    check=False, env=os.environ.copy(),
)
if _proc.returncode != 0:
    print(f'\n[ERROR] run_colab_bench.sh exited with code {_proc.returncode}.')
    print('        Inspect logs at /content/drive/MyDrive/mPES/_benchmark/')

In [ ]:
# pylint: disable=missing-module-docstring,reimported,ungrouped-imports,wrong-import-position,consider-using-with
import json     # noqa: E402
import os       # noqa: E402

# ============================================================================
# Cell 5 — INSPECT RESULTS
# ============================================================================
RESULTS_DRIVE = '/content/drive/MyDrive/mPES/_benchmark/results'
summary_path  = os.path.join(RESULTS_DRIVE, 'matrix_summary.json')
report_path   = os.path.join(RESULTS_DRIVE, 'benchmark_report.md')
if os.path.isfile(summary_path):
    with open(summary_path, encoding='utf-8') as _fh:
        s = json.load(_fh)
    print(f"models = {s['models']}")
    print(f"scenarios ({len(s['scenarios'])}) = {s['scenarios']}")
    n_done = sum(len(v) for v in s['cells'].values())
    print(f'cells completed: {n_done} / {len(s["models"]) * len(s["scenarios"])}')
if os.path.isfile(report_path):
    from IPython.display import Markdown, display  # type: ignore[import-not-found]
    with open(report_path, encoding='utf-8') as _fh:
        display(Markdown(_fh.read()))